# Stops, transfers and shifts of visits: demo

Concept: `overview/concepts/Visit Mappings.md` (outside this repo), section "identifying shifts" (with DD-S1).
Code: module `ocelescope_module_location_based_ocpm.visits.shifts` (and `.pipeline`); this notebook only imports and calls their functions.
The visits come from `visits.ipynb` (maximal visits); further detail: the docstrings (e.g. `help(transfers)`) and `docs/resources/shifts.md`.

What the notebook does:
1. loads the running example (Level 3) and the small edge-case example and derives their maximal visits,
2. finds the stops and transfers of every visit step by step and categorises the transfers (shifts), with their durations,
3. shows the option of DD-S1 (vehicle trips outside $hl$ as external shifts),
4. runs everything with one call (`analyse_visits`),
5. goes through the edge cases of shifts on the small example.

Start Jupyter from the repository root with

```
uv run --with jupyter jupyter lab notebooks/shifts.ipynb
```

Conventions in all frames: one row per hu-event relation $(e, hu)$ unless stated otherwise; `null` is $\bot$; durations are polars durations (e.g. `10h`).
All functions take and return polars frames; they work on eager (`DataFrame`) and lazy (`LazyFrame`) frames alike.

In [ ]:
from pathlib import Path

import polars as pl

from ocelescope_module_location_based_ocpm.derivation.level_files import MAPPING_SCHEMA, load_level_mapping
from ocelescope_module_location_based_ocpm.visits.context import location_context
from ocelescope_module_location_based_ocpm.visits.maximal import derive_maximal_visits, visit_relations
from ocelescope_module_location_based_ocpm.visits.pipeline import analyse_visits
from ocelescope_module_location_based_ocpm.visits.shifts import add_stops, shift_summary, transfers

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
EXAMPLE = ROOT / "data" / "running_example"
DERIVED = EXAMPLE / "derived"
EDGE = ROOT / "data" / "visits_example"

# Show whole frames.
pl.Config.set_tbl_rows(60)
pl.Config.set_tbl_cols(-1)
pl.Config.set_tbl_width_chars(250)
pl.Config.set_fmt_table_cell_list_len(8)
pl.Config.set_fmt_str_lengths(40);

## 1. Inputs

### Running example

The location mapping of Level 3, as saved by `location_derivation.ipynb` (`load_level_mapping(folder, level)`), and the location context from `data/running_example/inputs/locations.csv` (`location_context(table)`, see `trace_figure.ipynb`, section 1): $agg$ maps A1, A2 → A and B1 → B; T is not mapped, so it is its own high-level location; $V = \{T\}$, storage units $W = \{B1\}$.

The excursion tolerance is $\kappa_{max} = 2$, as in the note's figure of the maximal visits.

The maximal visits and their relations come from `derive_maximal_visits` and `visit_relations` (see `visits.ipynb`).

In [ ]:
LEVEL, KAPPA = 3, 2
mapping = load_level_mapping(DERIVED, LEVEL)
context = location_context(pl.read_csv(EXAMPLE / "inputs" / "locations.csv"))
rel, visits = derive_maximal_visits(mapping, "loc", context.agg, KAPPA)
vrel = visit_relations(rel, visits)
visits.select("visit_id", "hu", "high_level", "first_event", "last_event", "exit_event", "duration")

### Small edge-case example

`data/visits_example/` holds a small example made for these notebooks (generated by `make_visits_example.py`, the single source of its files): eight handling units h1 to h8 whose traces cover the edge cases.
The relations are in the format of a saved level mapping (read with the same schema `MAPPING_SCHEMA`); `loc` is given directly, as if derived.

Location context: A1, A2, the **crane C1** (a vehicle mapped *into* A) and the **storage units S1, S2** lie in hall A, B1 and the storage unit B2 in hall B, and the **truck T** is not mapped (a vehicle with its own high-level location); $V = \{C1, T\}$, $W = \{S1, S2, B2\}$.
Processing operations $P$ are given explicitly in `processing_events.csv`; three of them are movements ($P \cap M \neq \emptyset$).
The example is meant for $\kappa_{max} = 1$.

| handling unit | edge cases of shifts (section 5) |
|---|---|
| h1 | all four transfer categories in one visit; vehicle shift via a vehicle inside (C1) and outside (T) of the high-level location |
| h2 | vehicle relations at the start and the end of a visit (not shifts); vehicle visits |
| h3 | ⊥ inside a transfer (external shift); ⊥ visits have no stops |
| h4 | a transferring visit (no stops, but a relation not on a vehicle) |

In [ ]:
EDGE_KAPPA = 1
edge_relations = pl.read_csv(EDGE / "relations.csv", schema=MAPPING_SCHEMA)
edge_context = location_context(pl.read_csv(EDGE / "locations.csv"))
edge_processing = pl.read_csv(EDGE / "processing_events.csv")["event"].to_list()
print(edge_context)
print("P =", edge_processing)
edge_relations

## 2. Stops, transfers and shifts, step by step

Concept ("identifying shifts"): the **stops** of a visit $\nu = (hu, hl, \sigma)$, $hl \in HL$, are its relations at $hl$ that are not on a vehicle.
Between two consecutive stops $a_j$ and $a_{j+1}$ lies the **transfer** $\mu_j = otrace(hu)[a_j + 1 : a_{j+1} - 1]$, possibly empty, with one category:

| category | condition |
|---|---|
| stay | $\mu_j = \langle\rangle$ and both stops have the same $loc$ |
| direct shift | $\mu_j = \langle\rangle$ and the stops have different $loc$ |
| vehicle shift | $\mu_j \neq \langle\rangle$ and all relations of $\mu_j$ are on vehicles |
| external shift | $\mu_j \neq \langle\rangle$ and some relation of $\mu_j$ is not on a vehicle (⊥ counts as not on a vehicle) |

Direct and vehicle shifts are internal shifts. DD-S1: a trip on vehicles outside $hl$ (e.g. a truck not mapped to $hl$) is a vehicle shift by default (section 3 for the option). Vehicle relations before the first and after the last stop are entry and exit movements, not shifts. Visits at ⊥ have no stops.

### Stops: `add_stops(visit_relations, vehicles=(), location="loc")`

Adds `on_vehicle` ($loc \in V$; false for ⊥) and `is_stop` (at the visit's $hl \in HL$ and not on a vehicle).

| parameter | meaning | default |
|---|---|---|
| `visit_relations` | output of `visit_relations` | |
| `vehicles` | the set $V$ | empty |
| `location` | column of $loc$ | `"loc"` |

Example (note, table for $m_5$): c2's visit to A (visit 4) has stops at positions 1, 2, 3, 5, 6; e8 at T is on a vehicle.

In [ ]:
vrel = add_stops(vrel, context.vehicles)
vrel.filter(pl.col("visit_id") == 4).select("visit_id", "event", "position", "loc", "high_level", "at_location", "on_vehicle", "is_stop")

### Transfers: `transfers(visit_relations, location="loc", vehicles_outside_external=False)`

One row per transfer $\mu_j$, $j = 1, \dots, m - 1$:

| column | meaning |
|---|---|
| `visit_id`, `hu`, `visit_high_level`, `j` | the visit and the number of the transfer |
| `from_position`, `to_position`, `from_event`, `to_event`, `from_loc`, `to_loc` | the stops $a_j$ and $a_{j+1}$ |
| `transfer_events` | the events of $\mu_j$ in trace order (empty list for $\langle\rangle$) |
| `transfer_length` | $\lvert\mu_j\rvert = a_{j+1} - a_j - 1$; 0 for adjacent stops (the reversed index range of the note) |
| `all_on_vehicle` | all relations of $\mu_j$ on vehicles (true for $\langle\rangle$) |
| `all_at_location` | all relations of $\mu_j$ at the visit's $hl$ (true for $\langle\rangle$; used by DD-S1) |
| `transfer_duration` | $d(\mu_j) = time(\sigma[a_{j+1}]) - time(\sigma[a_j + 1])$: leaving the stop with the first event of $\mu_j$ until arriving at the next stop; 0 for $\langle\rangle$ |
| `category` | stay, direct shift, vehicle shift, external shift |

| parameter | meaning | default |
|---|---|---|
| `visit_relations` | output of `add_stops` | |
| `location` | column of $loc$ | `"loc"` |
| `vehicles_outside_external` | DD-S1 option (section 3) | `False` |

How it works: the stops are numbered $j$ in trace order and paired with the next stop (`shift(-1)`); every other relation gets the number of stops before it (cumulative sum), so the relations of $\mu_j$ are the non-stops with number $j$; they are aggregated per (visit, $j$) and joined to the pairs.
Relations before the first stop (number 0) and after the last stop (number $m$) belong to no transfer.

Example (note): c2's visit 4 = $m_5$: stay, direct shift (A1 → A2), vehicle shift via ⟨e8⟩ on T (although A2 → A2; $d = time(e9) - time(e8) = 1$ h), stay; c1's visit 1 = $m_2$: stay, direct, external via ⟨e7 at B1, e8 at T⟩ ($d = time(e9) - time(e7) = 2$ h), stay.

In [ ]:
transfer_frame = transfers(vrel)
transfer_frame

### Per visit: `shift_summary(visit_relations, transfers_frame, location="loc")`

One row per visit:

| column | meaning |
|---|---|
| `n_stops` | $m$ |
| `n_stays`, `n_direct_shifts`, `n_vehicle_shifts`, `n_external_shifts` | transfers per category; they add up to $m - 1$ |
| `n_internal_shifts` | direct + vehicle shifts |
| `internal_shift_time`, `external_shift_time` | total duration of the internal and of the external shifts |
| `locations`, `locations_at_hl`, `stop_locations`, `non_vehicle_locations` | $L(\nu)$, $L_{hl}(\nu)$, $L_{stop}(\nu)$, $L_{\neg v}(\nu)$ (sorted lists, ⊥ left out) |
| `is_vehicle_visit` | $L_{\neg v}(\nu) = \emptyset$ |
| `is_transferring_visit` | no stops, but $L_{\neg v}(\nu) \neq \emptyset$ |

For visits at ⊥, the stop and shift columns and both properties are null (only defined for $hl \in HL$).

Example (note): $L(m_2) = \{A1, A2, B1, T\}$, $L_{stop}(m_2) = \{A1, A2\}$; vehicle visits $m_6$ and $m_8$ (visits 5 and 7); $m_4$ (visit 3, c1 at T) is a transferring visit.

In [ ]:
shifts = shift_summary(vrel, transfer_frame)
shifts

## 3. Option (DD-S1): vehicle trips outside $hl$ as external shifts

By default, a transfer only on vehicles is a vehicle shift wherever the vehicles are, since a pickup may just be a way to get somewhere, and using vehicles as locations should not count as leaving $hl$.
With `vehicles_outside_external=True` (in `transfers` and `analyse_visits`), a transfer with at least one relation outside $hl$ (on a vehicle or not) is an **external shift**; only trips on vehicles mapped into $hl$ stay vehicle shifts.

Example: c2's $m_5$: the trip ⟨e8⟩ on the truck T (not mapped to A) becomes an external shift.

In [ ]:
transfers(vrel, vehicles_outside_external=True).filter(pl.col("visit_id") == 4).select(
    "visit_id", "j", "from_event", "to_event", "transfer_events", "all_on_vehicle", "all_at_location", "category"
)

## 4. All in one call: `analyse_visits(...)`

`analyse_visits` (see `visits.ipynb`, section 4) derives the maximal visits and runs `add_stops`, `transfers` and `shift_summary`; its `visits` frame carries the shift columns, `transfers` the transfers. The option of section 3 is `vehicles_outside_external`.

In [ ]:
analysis = analyse_visits(mapping, "loc", context.agg, context.vehicles, kappa=KAPPA)
print("same transfers as step by step:", analysis.transfers.equals(transfer_frame))
analysis.visits.select("visit_id", "hu", "high_level", *shifts.drop("visit_id").columns)

## 5. Edge cases on the small example

The small example with $\kappa_{max} = 1$.

In [ ]:
edge = analyse_visits(
    edge_relations, "loc", edge_context.agg, edge_context.vehicles, kappa=EDGE_KAPPA, processing=edge_processing,
)

### 5.1 All four transfer categories, vehicles inside and outside $hl$ (h1)

h1 at A (e1–e10): A1, A1, A2, C1, A2, T, A1, B1, A2, A2.
- e1 → e2: adjacent, same place: **stay**;
- e2 → e3: adjacent, A1 → A2: **direct shift**;
- e3 → e5 via ⟨e4 on C1⟩: C1 is a vehicle *inside* A (so e4 is at A, but not a stop): **vehicle shift**, although A2 → A2 ("picked up and put back");
- e5 → e7 via ⟨e6 on T⟩: T is a vehicle *outside* A, so e6 uses up the tolerance (gap 1 ≤ κ = 1): **vehicle shift** as well; the category does not depend on whether a vehicle is mapped into $hl$;
- e7 → e9 via ⟨e8 at B1⟩: **external shift**;
- e9 → e10: **stay**.

With `vehicles_outside_external=True` (second table), the trip via T (outside A) becomes an external shift; the one via C1 (in A) stays a vehicle shift.

In [ ]:
h1 = edge.visits.filter((pl.col("hu") == "h1") & (pl.col("high_level") == "A"))["visit_id"]
edge.transfers.filter(pl.col("visit_id").is_in(h1.implode())).drop("visit_high_level")

In [ ]:
edge_option = analyse_visits(
    edge_relations, "loc", edge_context.agg, edge_context.vehicles, kappa=EDGE_KAPPA, processing=edge_processing,
    vehicles_outside_external=True,
)
edge_option.transfers.filter(pl.col("visit_id").is_in(h1.implode())).select("j", "from_event", "to_event", "transfer_events", "all_at_location", "category")

### 5.2 Vehicles at the start and end of a visit, vehicle visits (h2)

h2: T, C1, A1, A1, C1, T, T. The visit to A runs from e13 (picked up from the truck by the crane C1, which is in A) to e16 (picked up again by C1).
Its stops are e14 and e15 only; the crane relations before the first and after the last stop are entry and exit movements, so the visit has one stay and no shift.
Both visits to T contain only relations on vehicles: **vehicle visits**. The last one ends with the trace, so its duration (e17 to e18) is a lower bound (`ends_trace`).

In [ ]:
edge.visits.filter(pl.col("hu") == "h2").select(
    "visit_id", "high_level", "first_event", "last_event", "exit_event", "ends_trace", "duration",
    "n_stops", "n_stays", "n_internal_shifts", "locations", "non_vehicle_locations", "is_vehicle_visit",
)

### 5.3 ⊥ (h3)

h3: ⊥, ⊥, A1, ⊥, A1, T, A2 (unknown locations in between, as at Level 0).
- ⊥ visits have no stops or shifts (null).
- The visit to A (e21–e25) contains e22 at ⊥: the transfer e21 → e23 via ⟨e22⟩ is an **external shift** (⊥ counts as not on a vehicle, "unknown" is the cautious choice); e23 → e25 via ⟨e24 on T⟩ is a vehicle shift.
- `locations` leaves ⊥ out.

In [ ]:
edge.visits.filter(pl.col("hu") == "h3").select(
    "visit_id", "high_level", "first_event", "last_event", "n_stops", "n_external_shifts", "n_vehicle_shifts",
    "locations", "is_vehicle_visit",
)

In [ ]:
h3 = edge.visits.filter(pl.col("hu") == "h3")["visit_id"]
edge.transfers.filter(pl.col("visit_id").is_in(h3.implode())).drop("visit_high_level")

### 5.4 Transferring visit (h4)

h4: T, A1, T, T, B1, B1, A2. The visit to T (e26–e29) contains the excursion ⟨e27 at A1⟩ (gap 1 ≤ κ). T is a vehicle, so the visit has no stops, but A1 is not a vehicle: a **transferring visit** (not a vehicle visit).

In [ ]:
edge.visits.filter(pl.col("hu") == "h4").select(
    "visit_id", "high_level", "first_event", "last_event", "n_stops", "locations", "non_vehicle_locations",
    "is_vehicle_visit", "is_transferring_visit",
)

### 5.5 Check

For every visit with at least one stop, the transfers add up to the number of stops minus one.

In [ ]:
for name, result in (("running example", analysis), ("edge-case example", edge)):
    with_stops = result.visits.filter(pl.col("n_stops") > 0)
    n_transfers = with_stops.select(pl.sum_horizontal("n_stays", "n_direct_shifts", "n_vehicle_shifts", "n_external_shifts"))
    print(f"{name}: transfers = stops - 1: {n_transfers.to_series().equals(with_stops['n_stops'] - 1, check_names=False)}")